In [ ]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## 1. Load dataset & pilih fitur numerik

In [ ]:
df = pd.read_csv('/home/hatch/workspace/user/files/MLWeek11_Dataset/songs_with_attributes_and_lyrics.csv')  
print(df.shape)
df.head()

(955320, 17)


,id,name,album_name,artists,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,lyrics
0,0Prct5TDjAnEgIqbxcldY9,!,UNDEN!ABLE,HELLYEAH,0.415,0.6050,7,-11.157,1,0.0575,0.00116,0.838000,0.4710,0.193,100.059,79500.0,"He said he came from Jamaica,\n he owned a cou..."
1,2ASl4wirkeYm3OWZxXKYuq,!!,NaN,Yxngxr1,0.788,0.6480,7,-9.135,0,0.3150,0.90000,0.000000,0.1760,0.287,79.998,114000.0,"Fucked a bitch, now she running with my kids\n..."
2,69lcggVPmOr9cvPx9kLiiN,!!! - Interlude,Where I Belong EP,Glowie,0.000,0.0354,7,-20.151,0,0.0000,0.90800,0.000000,0.4790,0.000,0.000,11413.0,"Oh, my God, I'm going crazy\n"
3,4U7dlZjg1s9pjdppqZy0fm,!!De Repente!!,Un Palo Al Agua (20 Grandes Canciones),Rosendo,0.657,0.8820,5,-6.340,1,0.0385,0.00740,0.000013,0.0474,0.939,123.588,198173.0,Continuamente se extraña la gente si no puede ...
4,4v1IBp3Y3rpkWmWzIlkYju,!!De Repente!!,Fuera De Lugar,Rosendo,0.659,0.8930,5,-8.531,1,0.0411,0.09220,0.000019,0.0534,0.951,123.600,199827.0,Continuamente se extraña la gente si no puede ...


In [ ]:
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

df = df.dropna(subset=features).reset_index(drop=True)
X = df[features].values
print("Jumlah track:", X.shape[0], "| Dimensi:", X.shape[1])

Jumlah track: 955320 | Dimensi: 9


## 2. Standarisasi fitur (StandardScaler)

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')
n, dim = X_scaled.shape
k = 10  # jumlah nearest neighbors

## 3. Pilih query track
Track query diambil dari dataset; track itu sendiri akan muncul sebagai tetangga ke-1 (jarak 0), sehingga kita ambil `k+1` hasil lalu membuang dirinya sendiri.

In [ ]:
query_idx = 1000  
query_vec = X_scaled[query_idx:query_idx+1]
print("Query track:", df.loc[query_idx, 'name'], "-", df.loc[query_idx, 'artists'])

def show(indices, dists=None, title=''):
    out = df.loc[indices, ['name', 'artists']].copy()
    if dists is not None:
        out['distance'] = np.round(dists, 4)
    print(f"\n=== {title} ===")
    display(out.reset_index())

Query track: (Come 'Round Here) I'm The One You Need - Smokey Robinson & The Miracles


## 4. Exact NN (brute force) – ground truth

In [6]:
t0 = time.time()
nn = NearestNeighbors(n_neighbors=k+1, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
build_exact = time.time() - t0

t0 = time.time()
d_ex, i_ex = nn.kneighbors(query_vec)
query_exact = time.time() - t0

idx_exact = [i for i in i_ex[0] if i != query_idx][:k]
dist_exact = [d for i, d in zip(i_ex[0], d_ex[0]) if i != query_idx][:k]
show(idx_exact, dist_exact, 'Exact NN')


=== Exact NN ===


,index,name,artists,distance
0,434574,La Vida del Leon - En Vivo,Alta Consigna,0.4901
1,190385,Dia Branco - Ao Vivo,Alceu Valença;Orquestra Ouro Preto,0.5020
2,878270,What the Hell Is Goin' on,"Elvin Bishop, Kid Andersen",0.5475
3,625534,Quand le rêve est fini,Plume Latraverse anf The Plumettes,0.5561
4,617507,Pretty Flamingo,Manfred Mann,0.5627
5,617510,Pretty Flamingo,Manfred Mann,0.5627
6,141347,Cien Kilos de Barro (100 Pounds of Clay),Enrique Guzman,0.5746
7,238003,Estoy Enamorado de Ti,Juan Gabriel,0.5823
8,128572,Carry Me,Trout Fishing in America,0.6013
9,752796,Take Me For Longing - Live From The Louisville...,Alison Krauss & Union Station,0.6553


## 5. ANNOY

In [7]:
t0 = time.time()
ann = AnnoyIndex(dim, 'euclidean')
for i in range(n):
    ann.add_item(i, X_scaled[i])
ann.build(20)  # jumlah trees
build_annoy = time.time() - t0

t0 = time.time()
res, dst = ann.get_nns_by_vector(query_vec[0], k+1, search_k=-1, include_distances=True)
query_annoy = time.time() - t0

pairs = [(i, d) for i, d in zip(res, dst) if i != query_idx][:k]
idx_annoy = [p[0] for p in pairs]
show(idx_annoy, [p[1] for p in pairs], 'ANNOY')


=== ANNOY ===


,index,name,artists,distance
0,434574,La Vida del Leon - En Vivo,Alta Consigna,0.4901
1,190385,Dia Branco - Ao Vivo,Alceu Valença;Orquestra Ouro Preto,0.5020
2,878270,What the Hell Is Goin' on,"Elvin Bishop, Kid Andersen",0.5475
3,625534,Quand le rêve est fini,Plume Latraverse anf The Plumettes,0.5561
4,617507,Pretty Flamingo,Manfred Mann,0.5627
5,617510,Pretty Flamingo,Manfred Mann,0.5627
6,141347,Cien Kilos de Barro (100 Pounds of Clay),Enrique Guzman,0.5746
7,238003,Estoy Enamorado de Ti,Juan Gabriel,0.5823
8,128572,Carry Me,Trout Fishing in America,0.6013
9,752796,Take Me For Longing - Live From The Louisville...,Alison Krauss & Union Station,0.6553


## 6. FAISS

In [8]:
# FAISS exact (referensi)
t0 = time.time()
faiss_flat = faiss.IndexFlatL2(dim)
faiss_flat.add(X_scaled)
build_flat = time.time() - t0
t0 = time.time()
faiss_flat.search(query_vec, k+1)
query_flat = time.time() - t0

# FAISS IVF (approximate)
nlist = int(4 * np.sqrt(n))
t0 = time.time()
quantizer = faiss.IndexFlatL2(dim)
faiss_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
faiss_ivf.train(X_scaled)
faiss_ivf.add(X_scaled)
build_faiss = time.time() - t0

faiss_ivf.nprobe = 10
t0 = time.time()
D, I = faiss_ivf.search(query_vec, k+1)
query_faiss = time.time() - t0

pairs = [(i, np.sqrt(d)) for i, d in zip(I[0], D[0]) if i != query_idx and i != -1][:k]
idx_faiss = [p[0] for p in pairs]
show(idx_faiss, [p[1] for p in pairs], 'FAISS (IVFFlat)')


=== FAISS (IVFFlat) ===


,index,name,artists,distance
0,434574,La Vida del Leon - En Vivo,Alta Consigna,0.4901
1,190385,Dia Branco - Ao Vivo,Alceu Valença;Orquestra Ouro Preto,0.5020
2,878270,What the Hell Is Goin' on,"Elvin Bishop, Kid Andersen",0.5475
3,625534,Quand le rêve est fini,Plume Latraverse anf The Plumettes,0.5561
4,617507,Pretty Flamingo,Manfred Mann,0.5627
5,617510,Pretty Flamingo,Manfred Mann,0.5627
6,141347,Cien Kilos de Barro (100 Pounds of Clay),Enrique Guzman,0.5746
7,238003,Estoy Enamorado de Ti,Juan Gabriel,0.5823
8,128572,Carry Me,Trout Fishing in America,0.6013
9,752796,Take Me For Longing - Live From The Louisville...,Alison Krauss & Union Station,0.6553


## 7. HNSW (hnswlib)

In [9]:
t0 = time.time()
hnsw = hnswlib.Index(space='l2', dim=dim)
hnsw.init_index(max_elements=n, ef_construction=200, M=16)
hnsw.add_items(X_scaled, np.arange(n))
build_hnsw = time.time() - t0

hnsw.set_ef(100)
t0 = time.time()
L, D = hnsw.knn_query(query_vec, k=k+1)
query_hnsw = time.time() - t0

pairs = [(i, np.sqrt(d)) for i, d in zip(L[0], D[0]) if i != query_idx][:k]
idx_hnsw = [p[0] for p in pairs]
show(idx_hnsw, [p[1] for p in pairs], 'HNSW')


=== HNSW ===


,index,name,artists,distance
0,434574,La Vida del Leon - En Vivo,Alta Consigna,0.4901
1,190385,Dia Branco - Ao Vivo,Alceu Valença;Orquestra Ouro Preto,0.5020
2,878270,What the Hell Is Goin' on,"Elvin Bishop, Kid Andersen",0.5475
3,625534,Quand le rêve est fini,Plume Latraverse anf The Plumettes,0.5561
4,617507,Pretty Flamingo,Manfred Mann,0.5627
5,617510,Pretty Flamingo,Manfred Mann,0.5627
6,141347,Cien Kilos de Barro (100 Pounds of Clay),Enrique Guzman,0.5746
7,238003,Estoy Enamorado de Ti,Juan Gabriel,0.5823
8,128572,Carry Me,Trout Fishing in America,0.6013
9,752796,Take Me For Longing - Live From The Louisville...,Alison Krauss & Union Station,0.6553


## 8. Perbandingan hasil (1 query)

In [10]:
def recall(pred, gt):
    return len(set(pred) & set(gt)) / len(gt)

summary = pd.DataFrame({
    'Metode': ['Exact NN', 'ANNOY', 'FAISS (IVFFlat)', 'HNSW'],
    'Build time (s)': [build_exact, build_annoy, build_faiss, build_hnsw],
    'Query time (ms)': [query_exact*1000, query_annoy*1000, query_faiss*1000, query_hnsw*1000],
    f'Recall@{k}': [1.0, recall(idx_annoy, idx_exact), recall(idx_faiss, idx_exact), recall(idx_hnsw, idx_exact)],
})
summary

,Metode,Build time (s),Query time (ms),Recall@10
0,Exact NN,0.151848,1009.318113,1.0
1,ANNOY,25.387285,13.302565,1.0
2,FAISS (IVFFlat),7.563875,0.269890,1.0
3,HNSW,157.677414,3.789902,1.0


In [ ]:
pd.DataFrame({
    'Exact': [df.loc[i, 'name'] for i in idx_exact],
    'ANNOY': [df.loc[i, 'name'] for i in idx_annoy],
    'FAISS': [df.loc[i, 'name'] for i in idx_faiss],
    'HNSW':  [df.loc[i, 'name'] for i in idx_hnsw],
}, index=range(1, k+1))

,Exact,ANNOY,FAISS,HNSW
1,La Vida del Leon - En Vivo,La Vida del Leon - En Vivo,La Vida del Leon - En Vivo,La Vida del Leon - En Vivo
2,Dia Branco - Ao Vivo,Dia Branco - Ao Vivo,Dia Branco - Ao Vivo,Dia Branco - Ao Vivo
3,What the Hell Is Goin' on,What the Hell Is Goin' on,What the Hell Is Goin' on,What the Hell Is Goin' on
4,Quand le rêve est fini,Quand le rêve est fini,Quand le rêve est fini,Quand le rêve est fini
5,Pretty Flamingo,Pretty Flamingo,Pretty Flamingo,Pretty Flamingo
6,Pretty Flamingo,Pretty Flamingo,Pretty Flamingo,Pretty Flamingo
7,Cien Kilos de Barro (100 Pounds of Clay),Cien Kilos de Barro (100 Pounds of Clay),Cien Kilos de Barro (100 Pounds of Clay),Cien Kilos de Barro (100 Pounds of Clay)
8,Estoy Enamorado de Ti,Estoy Enamorado de Ti,Estoy Enamorado de Ti,Estoy Enamorado de Ti
9,Carry Me,Carry Me,Carry Me,Carry Me
10,Take Me For Longing - Live From The Louisville...,Take Me For Longing - Live From The Louisville...,Take Me For Longing - Live From The Louisville...,Take Me For Longing - Live From The Louisville...


## 9. Evaluasi pada banyak query 

In [ ]:
rng = np.random.default_rng(42)
q_ids = rng.choice(n, 200, replace=False)
Q = X_scaled[q_ids]

t0 = time.time()
_, gt = nn.kneighbors(Q)
t_exact = (time.time() - t0) * 1000 / len(Q)
gt = [[j for j in row if j != q][:k] for row, q in zip(gt, q_ids)]

def avg_recall(preds):
    return np.mean([recall(p, g) for p, g in zip(preds, gt)])

# ANNOY
t0 = time.time()
r = [ann.get_nns_by_vector(q, k+1) for q in Q]
t_annoy = (time.time() - t0) * 1000 / len(Q)
p_annoy = [[j for j in row if j != qi][:k] for row, qi in zip(r, q_ids)]

# FAISS
t0 = time.time()
_, I = faiss_ivf.search(Q, k+1)
t_faiss = (time.time() - t0) * 1000 / len(Q)
p_faiss = [[j for j in row if j != qi][:k] for row, qi in zip(I, q_ids)]

# HNSW
t0 = time.time()
L, _ = hnsw.knn_query(Q, k=k+1)
t_hnsw = (time.time() - t0) * 1000 / len(Q)
p_hnsw = [[j for j in row if j != qi][:k] for row, qi in zip(L, q_ids)]

pd.DataFrame({
    'Metode': ['Exact NN', 'ANNOY', 'FAISS (IVFFlat)', 'HNSW'],
    'Rata-rata query time (ms)': [t_exact, t_annoy, t_faiss, t_hnsw],
    f'Rata-rata Recall@{k}': [1.0, avg_recall(p_annoy), avg_recall(p_faiss), avg_recall(p_hnsw)],
})

,Metode,Rata-rata query time (ms),Rata-rata Recall@10
0,Exact NN,7.987938,1.0000
1,ANNOY,0.149266,0.9540
2,FAISS (IVFFlat),0.392476,0.9855
3,HNSW,0.062561,0.9975
